In [ ]:
#Valve controller 
import sys
from PyQt5.QtWidgets import QApplication, QLabel, QMainWindow, QWidget, QGridLayout, QPushButton, QComboBox
from PyQt5.QtGui import * 
from qtwidgets import AnimatedToggle
from QSwitchControl import SwitchControl
from functools import partial
from PyQt5 import QtCore
import time
import keyboard
import serial
import re
from tqdm import tqdm
import numpy as np
import pandas as pd

import serial
import serial.tools.list_ports
import iqplot

In [ ]:
with serial.Serial(port, baudrate=115200, timeout=1) as arduino:
    handshake_arduino(arduino)

    # Flash the LEDs
    for _ in range(5):
        arduino.write(bytes([setVoltage]))
        sleep(1)

# OUTDATED CODE

In [1]:
#Valve controller 


import sys
from PyQt5.QtWidgets import *
from PyQt5.QtGui import * 
from qtwidgets import AnimatedToggle
from QSwitchControl import SwitchControl
from functools import partial
from PyQt5 import QtCore
import time
import keyboard
import serial
import re
from tqdm import tqdm
import numpy as np
import pandas as pd

import serial
import serial.tools.list_ports
import iqplot



def sleep(duration):
    """Sleep for `duration` seconds."""
    now = time.perf_counter()
    end = now + duration

    while now < end:
        now = time.perf_counter()

     
    
        
def handshake_arduino(arduino, sleep_time=1, print_handshake_message=False, handshake_code=0):
    """Make sure connection is established by sending
    and receiving bytes."""
    # Close and reopen
    arduino.close()
    arduino.open()

    # Chill out while everything gets set
    sleep(sleep_time)

    # Set a long timeout to complete handshake
    timeout = arduino.timeout
    arduino.timeout = 1

    # Read and discard everything that may be in the input buffer
    _ = arduino.read_all()

    # Send request to Arduino
    arduino.write(bytes([handshake_code]))

    # Read in what Arduino sent
    handshake_message = arduino.read_until()

    # Send and receive request again
    arduino.write(bytes([handshake_code]))
    handshake_message = arduino.read_until()

    # Print the handshake message, if desired
    if print_handshake_message:
        print("Handshake message: " + handshake_message.decode())

    # Reset the timeout
    arduino.timeout = timeout


valve_names = {1: 'A1', 2: 'A2', 3: 'A3', 4: 'A4', 5: 'A5', 6: 'A6', 7: 'A7', 8: 'A8',
    9: 'B1', 10: 'B2', 11: 'B3', 12: 'B4', 13: 'B5', 14: 'B6', 15: 'B7', 16: 'B8',
    17: 'C1', 18: 'C2', 19: 'C3', 20: 'C4', 21: 'C5', 22: 'C6', 23: 'C7', 24: 'C8',
    25: 'All A',
    26: 'All B',
    27: 'AllC',
    28: 'Valve 28',
    29: 'Valve 29',
    30: 'Valve 30'}

valve_switches = {0: 'A0', 1: 'A1', 2: 'A2', 3: 'A3', 4: 'A4', 5: 'A5', 6: 'A6', 7: 'A7', 8: 'A8',
        9: 'B1', 10: 'B2', 11: 'B3', 12: 'B4', 13: 'B5', 14: 'B6', 15: 'B7', 16: 'B8',
        17: 'C1', 18: 'C2', 19: 'C3', 20: 'C4', 21: 'C5', 22: 'C6', 23: 'C7', 24: 'C8',
        25: 'All A',
        26: 'All B',
        27: 'AllC'}



Grouped_valves = ['All A', 'All B', 'All C']

HANDSHAKE = 0
RED_LED_ON = 1
Voltage_ON = 2

port = 'COM3'
#arduino = serial.Serial(port, baudrate=115200, timeout=1)
setVoltage = 0

def set_voltage(set_value):
    setVoltage = int(set_value) + Voltage_ON
    print(setVoltage)
    return
  
    
def SendArduinoTrigger():
    with serial.Serial(port, baudrate=115200, timeout=1) as arduino:
        handshake_arduino(arduino)

    # Flash the LEDs
        for _ in range(10):
            arduino.write(bytes([2]))
            #sleep(1)
    

class stackedApp(QWidget):

    def __init__(self):
        super().__init__()
        self.leftlist = QListWidget ()
        self.leftlist.insertItem (0, 'Manual Actuation' )
        self.leftlist.insertItem (1, 'Experiment Automation' )

        self.stack1 = QWidget()
        self.stack2 = QWidget()

        self.stack1UI()
        self.stack2UI()

        self.Stack = QStackedWidget (self)
        self.Stack.addWidget (self.stack1)
        self.Stack.addWidget (self.stack2)

        hbox = QHBoxLayout(self)
        hbox.addWidget(self.leftlist)
        hbox.addWidget(self.Stack)

        self.setLayout(hbox)
        self.leftlist.currentRowChanged.connect(self.display)
        self.setGeometry(300, 50, 10,10)
        self.setWindowTitle('Valve and Electronics Controller')
        self.show()

    def stack1UI(self):
        layout = QGridLayout()
        self.stack1.setLayout(layout)
        
        label = QLabel("Cycling Period in minutes")
        cycle_edit = QLineEdit()
        layout.addWidget(label, 0, 0)
        layout.addWidget(cycle_edit, 1, 0, 1, 2)
        
        
        label = QLabel("Cycling Period in minutes")
        cycles_edit = QLineEdit()
        layout.addWidget(label, 0, 3)
        layout.addWidget(cycles_edit, 1, 3, 1, 2)
        
        
    def stack2UI(self):
        layout = QGridLayout()
        self.stack2.setLayout(layout)
        self.setStyleSheet("background-color: #f9f1e2;")
        self.setStyleSheet("QLabel{font-size: 15pt;}")
        My_Font = QFont("San Francisco", 12)
        #layout.setFont(My_Font)
        
        
        rows = 8
        columns = 4
        button_count = 30

        for i in range(24):
            valve_switches[i] = AnimatedToggle( checked_color="#68C16E", pulse_checked_color="#44FFB000")
            valve_name = valve_names[i+1]
            #valve_states[valve_name] = True
            valve_switches[i].setCheckable(True)
            valve_switches[i].setChecked(True)
            #valve_switches[i].stateChanged.connect(self.the_button_was_toggled)
            #button.clicked.connect(partial(update_valve_state, valve_name))
            label = QLabel(valve_names[i+1])
            layout.addWidget(label, (i % rows), 2 * (i // rows), alignment = QtCore.Qt.AlignmentFlag.AlignRight)
            layout.addWidget(valve_switches[i], (i % rows), 2 * (i // rows) + 1, alignment = QtCore.Qt.AlignmentFlag.AlignLeft)
        
        
        #ALL A
        valve_switches[25] = SwitchControl(bg_color="#777777", circle_color="#fcfbea", active_color="#d22ed1", animation_curve=QtCore.QEasingCurve.InOutCubic, animation_duration=100,  checked=False, change_cursor=False)
        #valve_switches[25].setCheckable(True)
        valve_switches[25].setChecked(False)
        valve_switches[25].stateChanged.connect(self.TurnA)
        label = QLabel(valve_names[25])
        layout.addWidget(label, 9, 0, alignment=QtCore.Qt.AlignmentFlag.AlignRight)
        layout.addWidget(valve_switches[25], 9, 1, alignment=QtCore.Qt.AlignmentFlag.AlignLeft)
        
        #ALL B
        valve_switches[26] = SwitchControl(bg_color="#777777", circle_color="#fcfbea", active_color="#d22ed1", animation_curve=QtCore.QEasingCurve.InOutCubic, animation_duration=100,  checked=False, change_cursor=False)
        #valve_switches[26].setCheckable(True)
        valve_switches[26].setChecked(False)
        valve_switches[26].stateChanged.connect(self.TurnB)
        label = QLabel(valve_names[26])
        layout.addWidget(label, 9, 2, alignment=QtCore.Qt.AlignmentFlag.AlignRight)
        layout.addWidget(valve_switches[26], 9, 3, alignment=QtCore.Qt.AlignmentFlag.AlignLeft)
        
        #ALL C
        valve_switches[27] = SwitchControl(bg_color="#777777", circle_color="#fcfbea", active_color="#d22ed1", animation_curve=QtCore.QEasingCurve.InOutCubic, animation_duration=100,  checked=False, change_cursor=False)
        #valve_switches[27].setCheckable(True)
        valve_switches[27].setChecked(False)
        valve_switches[27].stateChanged.connect(self.TurnC)
        label = QLabel('All C')
        layout.addWidget(label, 9, 4, alignment=QtCore.Qt.AlignmentFlag.AlignRight)
        layout.addWidget(valve_switches[27], 9, 5, alignment=QtCore.Qt.AlignmentFlag.AlignLeft)
            
        


        ############Pulse generator
        button1 = QPushButton('123')
        button1.setText("Generate Droplet")
        #button1.clicked.connect(generatePulse)
        layout.addWidget(button1, 12, 0, alignment=QtCore.Qt.AlignmentFlag.AlignRight)



        exit_button = QPushButton('Exit')
        exit_button.setStyleSheet("background-color:#e2757a;")
        #exit_button.clicked.connect(relayAllOn)
        exit_button.clicked.connect(QApplication.instance().quit)
        exit_button.clicked.connect(QApplication.closeAllWindows)
        exit_button.clicked.connect(QApplication.exit)
        layout.addWidget(exit_button, 12, 2, 1, 4)
        
        
        voltagevalues = QComboBox()
        voltagevalues.addItems(['100', '200', '300', '400', '500', '600', '700'])
        voltagevalues.currentTextChanged.connect(set_voltage)
        label = QLabel('Operating voltage in V')
        layout.addWidget(label, 0, 8)
        layout.addWidget(voltagevalues, 1, 8, 1, 3)
        
        self.TurnOnVolts = QPushButton('Volts OFF')
        self.TurnOnVolts.setStyleSheet("background-color: #bb283a;")
        self.TurnOnVolts.setCheckable(True)
        self.TurnOnVolts.setChecked(False)
        self.TurnOnVolts.clicked.connect(self.alter_state)
        layout.addWidget(self.TurnOnVolts, 3, 8, 1, 3)
        
        
        
        
        
    def alter_state(self, checked):
        if checked == True:
            self.TurnOnVolts.setText('Volts ON')
            self.TurnOnVolts.setStyleSheet("background-color: #2ac555;")
            SendArduinoTrigger()
        else:
            self.TurnOnVolts.setText('Volts OFF')
            self.TurnOnVolts.setStyleSheet("background-color: #bb283a;")
            
            
            
            
            
    def TurnA(self, checked):
        if checked == 2:
            for i in range(8):
                valve_switches[i].setChecked(True)
        elif checked == 0:
            for i in range(8):
                valve_switches[i].setChecked(False)
    
    
    def TurnB(self, checked):
        if checked == 2:
            for i in range(8,16):
                valve_switches[i].setChecked(True)
        elif checked == 0:
            for i in range(8,16):
                valve_switches[i].setChecked(False)
    
    
    def TurnC(self, checked):
        print('toggled')
        if checked == 2:
            for i in range(16,24):
                valve_switches[i].setChecked(True)
                
        elif checked == 0:
            for i in range(16,24):
                valve_switches[i].setChecked(False)
                
                
                
                
                
    def the_button_was_toggled(self, checked):
        print("Checked?", checked)


    def activate_tab_1(self):
        self.stacklayout.setCurrentIndex(0)

    def activate_tab_2(self):
        self.stacklayout.setCurrentIndex(1)
        
        
        
		
    def display(self,i):
      self.Stack.setCurrentIndex(i)
    
    
    
		
def main():
   app = QApplication(sys.argv)
   ex = stackedApp()
   sys.exit(app.exec_())
	
if __name__ == '__main__':
   main()

SystemExit: 0

C:\ProgramData\Anaconda3\envs\bootcamp\Lib\site-packages\IPython\core\interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


#                                                                 CURRENT CODE 

In [1]:
#Valve controller 
import sys
from PyQt5.QtWidgets import *
from PyQt5.QtGui import * 
from qtwidgets import AnimatedToggle
from QSwitchControl import SwitchControl
from functools import partial
from PyQt5 import QtCore
import time
import keyboard
import serial
import re
from tqdm import tqdm
import numpy as np
import pandas as pd

import serial
import serial.tools.list_ports
import iqplot



def sleep(duration):
    """Sleep for `duration` seconds."""
    now = time.perf_counter()
    end = now + duration

    while now < end:
        now = time.perf_counter()

     
    
        
def handshake_arduino(arduino, sleep_time=1, print_handshake_message=False, handshake_code=0):
    """Make sure connection is established by sending
    and receiving bytes."""
    # Close and reopen
    arduino.close()
    arduino.open()

    # Chill out while everything gets set
    sleep(sleep_time)

    # Set a long timeout to complete handshake
    timeout = arduino.timeout
    arduino.timeout = 1

    # Read and discard everything that may be in the input buffer
    _ = arduino.read_all()

    # Send request to Arduino
    arduino.write(bytes([handshake_code]))

    # Read in what Arduino sent
    handshake_message = arduino.read_until()

    # Send and receive request again
    arduino.write(bytes([handshake_code]))
    handshake_message = arduino.read_until()

    # Print the handshake message, if desired
    if print_handshake_message:
        print("Handshake message: " + handshake_message.decode())

    # Reset the timeout
    arduino.timeout = timeout


valve_names = {1: 'A1', 2: 'A2', 3: 'A3', 4: 'A4', 5: 'A5', 6: 'A6', 7: 'A7', 8: 'A8',
    9: 'B1', 10: 'B2', 11: 'B3', 12: 'B4', 13: 'B5', 14: 'B6', 15: 'B7', 16: 'B8',
    17: 'C1', 18: 'C2', 19: 'C3', 20: 'C4', 21: 'C5', 22: 'C6', 23: 'C7', 24: 'C8',
    25: 'All A',
    26: 'All B',
    27: 'AllC',
    28: 'Valve 28',
    29: 'Valve 29',
    30: 'Valve 30'}

valve_switches = {0: 'A0', 1: 'A1', 2: 'A2', 3: 'A3', 4: 'A4', 5: 'A5', 6: 'A6', 7: 'A7', 8: 'A8',
        9: 'B1', 10: 'B2', 11: 'B3', 12: 'B4', 13: 'B5', 14: 'B6', 15: 'B7', 16: 'B8',
        17: 'C1', 18: 'C2', 19: 'C3', 20: 'C4', 21: 'C5', 22: 'C6', 23: 'C7', 24: 'C8',
        25: 'All A',
        26: 'All B',
        27: 'AllC'}



Grouped_valves = ['All A', 'All B', 'All C']

HANDSHAKE = 0
RED_LED_ON = 1
Voltage_ON = 2

port = 'COM3'
#arduino = serial.Serial(port, baudrate=115200, timeout=1)
setVoltage = 0
Cycles = 0
CyclePeriod = 1

def set_voltage(set_value):
    setVoltage = int(set_value) + Voltage_ON
    print(setVoltage)
    return
  
    
def SendArduinoTrigger():
    with serial.Serial(port, baudrate=115200, timeout=1) as arduino:
        handshake_arduino(arduino)

    # Flash the LEDs
        for _ in range(10):
            arduino.write(bytes([2]))
            #sleep(1)
    

class stackedApp(QMainWindow):

    def __init__(self):
        super().__init__()
        tabs = QTabWidget()
        tabs.setTabPosition(QTabWidget.West)
        tabs.setMovable(True)

        self.stack1 = QWidget()
        self.stack2 = QWidget()

        self.stack1UI()
        self.stack2UI()

        tabs.addTab(self.stack1, "Experiment")
        tabs.addTab(self.stack2, "Manual")
        tabs.setStyleSheet("QLabel{font-size: 15pt;}")
        self.setCentralWidget(tabs)

        #hbox = QHBoxLayout(self)
        #hbox.addWidget(self.leftlist)
        #hbox.addWidget(self.Stack)
        #self.setCentralWidget(tabs)
        
        #self.setLayout(hbox)
        #self.leftlist.currentRowChanged.connect(self.display)
        #self.setGeometry(300, 50, 10,10)
        self.setWindowTitle('Valve and Electronics Controller')
        self.show()
        
        
        

    def stack1UI(self):
        layout = QGridLayout()
        self.stack1.setLayout(layout)
        
        label = QLabel("No. of Cycles")
        cycle_edit = QLineEdit()
        self.Cycles = 0
        cycle_edit.textChanged.connect(self.set_Cycles)
        layout.addWidget(label, 0, 0, alignment = QtCore.Qt.AlignmentFlag.AlignBottom)
        layout.addWidget(cycle_edit, 2, 0, alignment = QtCore.Qt.AlignmentFlag.AlignTop)
        
        
        label = QLabel("Cycling Period in seconds")
        period_edit = QLineEdit()
        self.CyclePeriod = 0.1
        period_edit.textChanged.connect(self.set_Period)
        layout.addWidget(label, 0, 4, alignment = QtCore.Qt.AlignmentFlag.AlignBottom)
        layout.addWidget(period_edit, 2, 4)
        
        self.pbar = QProgressBar()
        self.pbar.setGeometry(30, 40, 200, 25) 
        btn = QPushButton('Start')
        btn.clicked.connect(self.Start_experiment)
        layout.addWidget(btn, 5, 0)
        layout.addWidget(self.pbar, 5, 2)
        
        
        
        exit_button = QPushButton('Exit')
        exit_button.setStyleSheet("background-color:#e2757a;")
        #exit_button.clicked.connect(relayAllOn)
        exit_button.clicked.connect(QApplication.instance().quit)
        exit_button.clicked.connect(QApplication.closeAllWindows)
        exit_button.clicked.connect(QApplication.exit)
        layout.addWidget(exit_button, 12, 4, 1, 4)
        
        
    
    
    def set_Cycles(self, s):
        self.Cycles = int(s)
        self.pbar.setRange(0, self.Cycles)
        print(self.Cycles)
        
    
    def set_Period(self, s):
        self.CyclePeriod = float(s)
        print(self.CyclePeriod)

    
    
    def Start_experiment(self):
        for i in range(self.Cycles + 1):
            print(self.CyclePeriod)
            # slowing down the loop 
            sleep(self.CyclePeriod)
  
            self.pbar.setValue(i)
            print(i)
        
                
        
        
        
        
    def stack2UI(self):
        layout = QGridLayout()
        self.stack2.setLayout(layout)
        self.setStyleSheet("background-color: #f9f1e2;")
        self.setStyleSheet("QLabel{font-size: 15pt;}")
        My_Font = QFont("San Francisco", 12)
        
        rows = 8
        columns = 4
        button_count = 30

        for i in range(24):
            valve_switches[i] = AnimatedToggle( checked_color="#68C16E", pulse_checked_color="#44FFB000")
            valve_name = valve_names[i+1]
            #valve_states[valve_name] = True
            valve_switches[i].setCheckable(True)
            valve_switches[i].setChecked(True)
            #valve_switches[i].stateChanged.connect(self.the_button_was_toggled)
            #button.clicked.connect(partial(update_valve_state, valve_name))
            label = QLabel(valve_names[i+1])
            layout.addWidget(label, (i % rows), 2 * (i // rows), alignment = QtCore.Qt.AlignmentFlag.AlignRight)
            layout.addWidget(valve_switches[i], (i % rows), 2 * (i // rows) + 1, alignment = QtCore.Qt.AlignmentFlag.AlignLeft)
        
        
        #ALL A
        valve_switches[25] = SwitchControl(bg_color="#777777", circle_color="#fcfbea", active_color="#d22ed1", animation_curve=QtCore.QEasingCurve.InOutCubic, animation_duration=100,  checked=False, change_cursor=False)
        #valve_switches[25].setCheckable(True)
        valve_switches[25].setChecked(False)
        valve_switches[25].stateChanged.connect(self.TurnA)
        label = QLabel(valve_names[25])
        layout.addWidget(label, 9, 0, alignment=QtCore.Qt.AlignmentFlag.AlignRight)
        layout.addWidget(valve_switches[25], 9, 1, alignment=QtCore.Qt.AlignmentFlag.AlignLeft)
        
        #ALL B
        valve_switches[26] = SwitchControl(bg_color="#777777", circle_color="#fcfbea", active_color="#d22ed1", animation_curve=QtCore.QEasingCurve.InOutCubic, animation_duration=100,  checked=False, change_cursor=False)
        #valve_switches[26].setCheckable(True)
        valve_switches[26].setChecked(False)
        valve_switches[26].stateChanged.connect(self.TurnB)
        label = QLabel(valve_names[26])
        layout.addWidget(label, 9, 2, alignment=QtCore.Qt.AlignmentFlag.AlignRight)
        layout.addWidget(valve_switches[26], 9, 3, alignment=QtCore.Qt.AlignmentFlag.AlignLeft)
        
        #ALL C
        valve_switches[27] = SwitchControl(bg_color="#777777", circle_color="#fcfbea", active_color="#d22ed1", animation_curve=QtCore.QEasingCurve.InOutCubic, animation_duration=100,  checked=False, change_cursor=False)
        #valve_switches[27].setCheckable(True)
        valve_switches[27].setChecked(False)
        valve_switches[27].stateChanged.connect(self.TurnC)
        label = QLabel('All C')
        layout.addWidget(label, 9, 4, alignment=QtCore.Qt.AlignmentFlag.AlignRight)
        layout.addWidget(valve_switches[27], 9, 5, alignment=QtCore.Qt.AlignmentFlag.AlignLeft)
            
        


        ############Pulse generator
        button1 = QPushButton('123')
        button1.setText("Generate Droplet")
        #button1.clicked.connect(generatePulse)
        layout.addWidget(button1, 12, 0, alignment=QtCore.Qt.AlignmentFlag.AlignRight)



        exit_button = QPushButton('Exit')
        exit_button.setStyleSheet("background-color:#e2757a;")
        #exit_button.clicked.connect(relayAllOn)
        exit_button.clicked.connect(QApplication.instance().quit)
        exit_button.clicked.connect(QApplication.closeAllWindows)
        exit_button.clicked.connect(QApplication.exit)
        layout.addWidget(exit_button, 12, 2, 1, 4)
        
        
        voltagevalues = QComboBox()
        voltagevalues.addItems(['100', '200', '300', '400', '500', '600', '700'])
        voltagevalues.currentTextChanged.connect(set_voltage)
        label = QLabel('Operating voltage in V')
        layout.addWidget(label, 0, 8)
        layout.addWidget(voltagevalues, 1, 8, 1, 3)
        
        self.TurnOnVolts = QPushButton('Volts OFF')
        self.TurnOnVolts.setStyleSheet("background-color: #bb283a;")
        self.TurnOnVolts.setCheckable(True)
        self.TurnOnVolts.setChecked(False)
        self.TurnOnVolts.clicked.connect(self.alter_state)
        layout.addWidget(self.TurnOnVolts, 3, 8, 1, 3)
        
    
    
    

    def alter_state(self, checked):
        if checked == True:
            self.TurnOnVolts.setText('Volts ON')
            self.TurnOnVolts.setStyleSheet("background-color: #2ac555;")
            SendArduinoTrigger()
        else:
            self.TurnOnVolts.setText('Volts OFF')
            self.TurnOnVolts.setStyleSheet("background-color: #bb283a;")
            
            
            
            
            
    def TurnA(self, checked):
        if checked == 2:
            for i in range(8):
                valve_switches[i].setChecked(True)
        elif checked == 0:
            for i in range(8):
                valve_switches[i].setChecked(False)
    
    
    def TurnB(self, checked):
        if checked == 2:
            for i in range(8,16):
                valve_switches[i].setChecked(True)
        elif checked == 0:
            for i in range(8,16):
                valve_switches[i].setChecked(False)
    
    
    def TurnC(self, checked):
        print('toggled')
        if checked == 2:
            for i in range(16,24):
                valve_switches[i].setChecked(True)
                
        elif checked == 0:
            for i in range(16,24):
                valve_switches[i].setChecked(False)
                
                
                
                
                
    def the_button_was_toggled(self, checked):
        print("Checked?", checked)


    def activate_tab_1(self):
        self.stacklayout.setCurrentIndex(0)

    def activate_tab_2(self):
        self.stacklayout.setCurrentIndex(1)
        
        
        
		
    def display(self,i):
        self.Stack.setCurrentIndex(i)
    
    
    
		
def main():
    app = QApplication(sys.argv)
    ex = stackedApp()
    sys.exit(app.exec_())
	
if __name__ == '__main__':
   main()

SystemExit: 0

C:\ProgramData\Anaconda3\envs\bootcamp\Lib\site-packages\IPython\core\interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
from random import randint

import pyqtgraph as pg
from PyQt5 import QtCore, QtWidgets

class MainWindow(QtWidgets.QMainWindow):
    def __init__(self):
        super().__init__()

        # Temperature vs time dynamic plot
        self.plot_graph = pg.PlotWidget()
        self.setCentralWidget(self.plot_graph)
        self.plot_graph.setBackground("w")
        pen = pg.mkPen(color=(255, 0, 0))
        self.plot_graph.setTitle("Temperature vs Time", color="b", size="20pt")
        styles = {"color": "red", "font-size": "18px"}
        self.plot_graph.setLabel("left", "Temperature (°C)", **styles)
        self.plot_graph.setLabel("bottom", "Time (min)", **styles)
        self.plot_graph.addLegend()
        self.plot_graph.showGrid(x=True, y=True)
        self.plot_graph.setYRange(20, 40)
        self.time = list(range(10))
        self.temperature = [randint(20, 40) for _ in range(10)]
        # Get a line reference
        self.line = self.plot_graph.plot(
            self.time,
            self.temperature,
            name="Temperature Sensor",
            pen=pen,
            symbol="+",
            symbolSize=15,
            symbolBrush="b",
        )
        # Add a timer to simulate new temperature measurements
        self.timer = QtCore.QTimer()
        self.timer.setInterval(300)
        self.timer.timeout.connect(self.update_plot)
        self.timer.start()

    def update_plot(self):
        #self.time = self.time[1:]
        self.time.append(self.time[-1] + 1)
        #self.temperature = self.temperature[1:]
        self.temperature.append(randint(20, 40))
        self.line.setData(self.time, self.temperature)

app = QtWidgets.QApplication([])
main = MainWindow()
main.show()
app.exec()

In [1]:
from random import randint

import pyqtgraph as pg
from PyQt5 import QtCore, QtWidgets

import numpy as np
from skimage import io
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np


img = io.imread('stack2/chem_1_6_004.tif')
vertices = [(74, 100),  # upper left corner
            (174, 135),  # lower right corner
            (74, 170), # and so on...
            (174, 202)]

chems = 2 # number of chemostats to be analyzed
cycles =  len(io.imread_collection('stack2/chem_1*.tif'))
avgflo = np.empty([chems, cycles])

for chem_iterator in range(chems):
    
    ic = io.imread_collection(f"stack2/chem_{chem_iterator+1}*.tif")

    for i in range(len(ic)):
    
        BFsubimg = ic[i][74:174, 100:135] #top:bottom, left:right
        darksubimg = ic[i][74:174, 170:202]
    
        avgflo[chem_iterator, i] = (BFsubimg.mean() - darksubimg.mean()) #average fluorescence intensity

    #print(avgflo)



class MainWindow(QtWidgets.QMainWindow):
    def __init__(self):
        super().__init__()

        # Temperature vs time plot
        self.plot_graph = pg.PlotWidget()
        self.setCentralWidget(self.plot_graph)
        self.plot_graph.setBackground("w")
        self.plot_graph.setTitle("Fluorescence vs Time", color="b", size="20pt")
        styles = {"color": "black", "font-size": "18px"}
        self.plot_graph.setLabel("left", "RFU)", **styles)
        self.plot_graph.setLabel("bottom", "Time (min)", **styles)
        self.plot_graph.addLegend()
        self.plot_graph.showGrid(x=True, y=True)
        #self.plot_graph.setXRange(1, 10)
        self.time = [1]
        self.j=0
        self.GFP_1 = [avgflo[0,self.j]]
        self.GFP_2 = [avgflo[1,self.j]]
        #self.plot_graph.setYRange(0, np.max(self.GFP_1, self.GFP_2))
        pen = pg.mkPen(color=(0, 255, 0), width=2)
        self.line1 = self.plot_graph.plot(
            self.time,
            self.GFP_1,
            name="Chemostat 1",
            pen=pen,
            symbol="+",
            symbolSize=4,
            symbolBrush="b",
        )
        pen = pg.mkPen(color=(255, 0, 0), width=2)
        self.line2 = self.plot_graph.plot(
            self.time,
            self.GFP_2,
            name="Chemostat 2",
            pen=pen,
            symbol="+",
            symbolSize=4,
            symbolBrush="b",
        )

        self.timer = QtCore.QTimer()
        self.timer.setInterval(300)
        self.timer.timeout.connect(self.update_plot)
        self.timer.start()

    def update_plot(self):
        #self.time = self.time[1:]
        self.time.append(self.time[-1] + 1)
        #self.temperature = self.temperature[1:]
        self.j = self.j + 1
        self.GFP_1.append(avgflo[0,self.j])
        self.GFP_2.append(avgflo[1,self.j])
        self.plot_graph.setYRange(0, np.max((self.GFP_1, self.GFP_2)))
        self.line1.setData(self.time, self.GFP_1)
        self.line2.setData(self.time, self.GFP_2)

app = QtWidgets.QApplication([])
main = MainWindow()
main.show()
app.exec()

0

In [ ]:
import numpy as np
from skimage import io
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

img = io.imread('stack2/chem_1_6_004.tif')

vertices = [(74, 100),  # upper left corner
            (174, 135),  # lower right corner
            (74, 170), # and so on...
            (174, 202)]

chems = 2 # number of chemostats to be analyzed
cycles =  len(io.imread_collection('stack2/chem_1*.tif'))
avgflo = np.empty([chems, cycles])

for chem_iterator in range(chems):
    
    ic = io.imread_collection(f"stack2/chem_{chem_iterator+1}*.tif")

    for i in range(len(ic)):
    
        BFsubimg = ic[i][74:174, 100:135] #top:bottom, left:right
        darksubimg = ic[i][74:174, 170:202]
    
        avgflo[chem_iterator, i] = (BFsubimg.mean() - darksubimg.mean()) #average fluorescence intensity

    print(avgflo)


plt.show()

In [9]:
import glob

from skimage import io
import ipympl
import matplotlib.pyplot as plt
import numpy as np
import cv2
import skimage as ski



def image_pipeline(givenimage):
    shapes01 = cv2.imread(givenimage, -1)

    #fig, ax = plt.subplots()
    #ax.imshow(shapes01)

    # blur the image to denoise
    blurred_shapes = ski.filters.gaussian(shapes01, sigma=1.0)

    #fig, ax = plt.subplots()

    t = ski.filters.threshold_otsu(blurred_shapes)
    #print("Found automatic threshold t = {}.".format(t))

    binary_mask = blurred_shapes > t

    #fig, ax = plt.subplots()
    #ax.imshow(binary_mask, cmap="gray")


    # get largest contour and draw it on input
    result = shapes01.copy()
    binary = np.asarray(binary_mask, dtype="uint8")
    contours = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = contours[0] if len(contours) == 2 else contours[1]
    big_contour = max(contours, key=cv2.contourArea)
    mask = np.zeros_like(shapes01)
    #print("area is", cv2.contourArea(big_contour))
    #result = np.asarray(shapes01, dtype="uint8")
    cv2.drawContours(binary, [big_contour], 0, (255,0,0), 2)
    out = np.zeros_like(shapes01) # Extract out the object and place into output image
    out[binary_mask == True] = shapes01[binary_mask == True]


    #print(big_contour)
    # save results
    cv2.imwrite('light_contour.tiff', binary)

    #cv2.imshow("result", result)
    io.imsave('Output.tiff', binary_mask)
    average_fluo = []
    for i in range(0, shapes01.shape[0]): # We go over rows number 
        for j in range(0, shapes01.shape[1]): # we go over columns number
            if out[i,j] != 0:
               average_fluo.append(out[i,j])

    mean_val = cv2.mean(shapes01, mask = binary)
    return(cv2.contourArea(big_contour), np.mean(average_fluo))
    #print(np.mean(average_fluo))
    #print( mean_val)

    
    
    
volume = []    
fluorescence = []    
    
for chem_iterator in range(133,231):    
    ic = f"dropletstack/test_0_10_2_00{chem_iterator}.tiff"
    vol,fluor = image_pipeline(ic)
    volume.append(vol)
    fluorescence.append(fluor)
print(volume, fluorescence)

C:\Users\amogh\AppData\Local\Temp\ipykernel_5992\3735482311.py:51: UserWarning: Output.tiff is a boolean image: setting True to 255 and False to 0. To silence this warning, please convert the image using img_as_ubyte.
  io.imsave('Output.tiff', binary_mask)


[20080.0, 20033.0, 19992.5, 19925.5, 19887.5, 19840.0, 19797.0, 19739.0, 19695.5, 19633.0, 19601.0, 19559.5, 19513.0, 19463.5, 19411.5, 19360.5, 19304.0, 19263.5, 19205.5, 19180.0, 19135.0, 19082.5, 19038.5, 18991.0, 18971.0, 18912.5, 18881.5, 18852.5, 18784.0, 18751.5, 18695.5, 18676.5, 18635.5, 18583.5, 18553.5, 18502.5, 18483.5, 18430.5, 18399.0, 18335.5, 18311.5, 18273.0, 18230.5, 18206.5, 18172.5, 18121.0, 18082.0, 18058.5, 18025.5, 17992.0, 17952.0, 17901.0, 17885.0, 17830.5, 17792.5, 17779.0, 17745.5, 17694.0, 17667.0, 17652.0, 17616.0, 17596.0, 17558.0, 17542.0, 17499.5, 17436.0, 17419.5, 17384.5, 17356.5, 17339.5, 17295.0, 17259.5, 17238.0, 17213.0, 17181.5, 17146.5, 17113.0, 17092.5, 17062.0, 17053.5, 17028.5, 17003.5, 16978.0, 16913.0, 16889.0, 16844.0, 16815.0, 16790.5, 16771.0, 16752.0, 16732.0, 16701.0, 16682.0, 16652.0, 16617.5, 16577.5, 16556.0, 16539.5] [9541.320711049833, 9521.9818370268, 9502.803264094955, 9490.346846623332, 9457.446283867761, 9444.076344246774, 9421

In [1]:
from random import randint

import pyqtgraph as pg
from PyQt5 import QtCore, QtWidgets
from skimage import io
import matplotlib.pyplot as plt
import matplotlib.patches as patches

#Valve controller 
import sys
from PyQt5.QtWidgets import *
from PyQt5.QtGui import * 
from qtwidgets import AnimatedToggle
from QSwitchControl import SwitchControl
from functools import partial
from PyQt5 import QtCore
import time
import keyboard
import serial
import re
from tqdm import tqdm
import numpy as np
import pandas as pd

import serial
import serial.tools.list_ports
import iqplot
import glob
import ipympl
import matplotlib.pyplot as plt
import cv2
import skimage as ski



def image_pipeline(givenimage):
    shapes01 = cv2.imread(givenimage, -1)

    #fig, ax = plt.subplots()
    #ax.imshow(shapes01)

    # blur the image to denoise
    blurred_shapes = ski.filters.gaussian(shapes01, sigma=1.0)

    #fig, ax = plt.subplots()

    t = ski.filters.threshold_otsu(blurred_shapes)
    #print("Found automatic threshold t = {}.".format(t))

    binary_mask = blurred_shapes > t

    #fig, ax = plt.subplots()
    #ax.imshow(binary_mask, cmap="gray")


    # get largest contour and draw it on input
    result = shapes01.copy()
    binary = np.asarray(binary_mask, dtype="uint8")
    contours = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = contours[0] if len(contours) == 2 else contours[1]
    big_contour = max(contours, key=cv2.contourArea)
    mask = np.zeros_like(shapes01)
    #print("area is", cv2.contourArea(big_contour))
    #result = np.asarray(shapes01, dtype="uint8")
    cv2.drawContours(binary, [big_contour], 0, (255,0,0), 2)
    out = np.zeros_like(shapes01) # Extract out the object and place into output image
    out[binary_mask == True] = shapes01[binary_mask == True]


    #print(big_contour)
    # save results
    cv2.imwrite('light_contour.tiff', binary)

    #cv2.imshow("result", result)
    io.imsave('Output.tiff', binary_mask)
    average_fluo = []
    for i in range(0, shapes01.shape[0]): # We go over rows number 
        for j in range(0, shapes01.shape[1]): # we go over columns number
            if out[i,j] != 0:
               average_fluo.append(out[i,j])

    mean_val = cv2.mean(shapes01, mask = binary)
    return(cv2.contourArea(big_contour), np.mean(average_fluo))
    #print(np.mean(average_fluo))
    #print( mean_val)

    
    
    
volume = [0]    
fluorescence = [0]    
    


def sleep(duration):
    """Sleep for `duration` seconds."""
    now = time.perf_counter()
    end = now + duration

    while now < end:
        now = time.perf_counter()

     
    
        
def handshake_arduino(arduino, sleep_time=1, print_handshake_message=False, handshake_code=0):
    """Make sure connection is established by sending
    and receiving bytes."""
    # Close and reopen
    arduino.close()
    arduino.open()

    # Chill out while everything gets set
    sleep(sleep_time)

    # Set a long timeout to complete handshake
    timeout = arduino.timeout
    arduino.timeout = 1

    # Read and discard everything that may be in the input buffer
    _ = arduino.read_all()

    # Send request to Arduino
    arduino.write(bytes([handshake_code]))

    # Read in what Arduino sent
    handshake_message = arduino.read_until()

    # Send and receive request again
    arduino.write(bytes([handshake_code]))
    handshake_message = arduino.read_until()

    # Print the handshake message, if desired
    if print_handshake_message:
        print("Handshake message: " + handshake_message.decode())

    # Reset the timeout
    arduino.timeout = timeout


valve_names = {1: 'A1', 2: 'A2', 3: 'A3', 4: 'A4', 5: 'A5', 6: 'A6', 7: 'A7', 8: 'A8',
    9: 'B1', 10: 'B2', 11: 'B3', 12: 'B4', 13: 'B5', 14: 'B6', 15: 'B7', 16: 'B8',
    17: 'C1', 18: 'C2', 19: 'C3', 20: 'C4', 21: 'C5', 22: 'C6', 23: 'C7', 24: 'C8',
    25: 'All A',
    26: 'All B',
    27: 'AllC',
    28: 'Valve 28',
    29: 'Valve 29',
    30: 'Valve 30'}

valve_switches = {0: 'A0', 1: 'A1', 2: 'A2', 3: 'A3', 4: 'A4', 5: 'A5', 6: 'A6', 7: 'A7', 8: 'A8',
        9: 'B1', 10: 'B2', 11: 'B3', 12: 'B4', 13: 'B5', 14: 'B6', 15: 'B7', 16: 'B8',
        17: 'C1', 18: 'C2', 19: 'C3', 20: 'C4', 21: 'C5', 22: 'C6', 23: 'C7', 24: 'C8',
        25: 'All A',
        26: 'All B',
        27: 'AllC'}



Grouped_valves = ['All A', 'All B', 'All C']

HANDSHAKE = 0
RED_LED_ON = 1
Voltage_ON = 2

port = 'COM3'
#arduino = serial.Serial(port, baudrate=115200, timeout=1)
setVoltage = 0
Cycles = 0
CyclePeriod = 1

def set_voltage(set_value):
    setVoltage = int(set_value) + Voltage_ON
    print(setVoltage)
    return
  
    
def SendArduinoTrigger():
    with serial.Serial(port, baudrate=115200, timeout=1) as arduino:
        handshake_arduino(arduino)

    # Flash the LEDs
        for _ in range(10):
            arduino.write(bytes([2]))
            #sleep(1)
    

class stackedApp(QMainWindow):

    def __init__(self):
        super().__init__()
        tabs = QTabWidget()
        tabs.setTabPosition(QTabWidget.West)
        tabs.setMovable(True)

        self.stack1 = QWidget()
        self.stack2 = QWidget()

        self.stack1UI()
        self.stack2UI()

        tabs.addTab(self.stack1, "Experiment")
        tabs.addTab(self.stack2, "Manual")
        tabs.setStyleSheet("QLabel{font-size: 15pt;}")
        self.setCentralWidget(tabs)

        #hbox = QHBoxLayout(self)
        #hbox.addWidget(self.leftlist)
        #hbox.addWidget(self.Stack)
        #self.setCentralWidget(tabs)
        
        #self.setLayout(hbox)
        #self.leftlist.currentRowChanged.connect(self.display)
        #self.setGeometry(300, 50, 10,10)
        self.setWindowTitle('Valve and Electronics Controller')
        self.show()
        
        
        

    def stack1UI(self): #Automated setup
        
        layout = QGridLayout()
        self.stack1.setLayout(layout)
        
        label = QLabel("No. of Cycles")
        cycle_edit = QLineEdit()
        self.Cycles = 0
        cycle_edit.textChanged.connect(self.set_Cycles)
        layout.addWidget(label, 0, 20, alignment = QtCore.Qt.AlignmentFlag.AlignBottom)
        layout.addWidget(cycle_edit, 2, 20, alignment = QtCore.Qt.AlignmentFlag.AlignTop)
        
        
        label = QLabel("Cycling Period in seconds")
        period_edit = QLineEdit()
        self.CyclePeriod = 0.1
        period_edit.textChanged.connect(self.set_Period)
        layout.addWidget(label, 0, 4, alignment = QtCore.Qt.AlignmentFlag.AlignBottom)
        layout.addWidget(period_edit, 2, 4)
        
        self.pbar = QProgressBar()
        self.pbar.setGeometry(30, 40, 200, 25) 
        btn = QPushButton('Start')
        btn.clicked.connect(self.Start_experiment)
        layout.addWidget(btn, 5, 0)
        layout.addWidget(self.pbar, 5, 2)
        
        
        
        self.plot_graph = pg.PlotWidget()
        layout.addWidget(self.plot_graph)
        self.plot_graph.setBackground("w")
        self.plot_graph.setTitle("Fluorescence vs Time", color="b", size="20pt")
        styles = {"color": "black", "font-size": "18px"}
        self.plot_graph.setLabel("left", "RFU", **styles)
        self.plot_graph.setLabel("bottom", "Time (min)", **styles)
        self.plot_graph.addLegend()
        self.plot_graph.showGrid(x=True, y=True)
        self.plot_graph.setYRange(0, np.max(fluorescence))
        #self.plot_graph.setXRange(1, 10)
        self.timestamp = [0]
        self.j=0
        pen = pg.mkPen(color=(0, 255, 0), width=2)
        self.line1 = self.plot_graph.plot(
            self.timestamp,
            fluorescence,
            name="Chemostat 1",
            pen=pen,
            symbol="+",
            symbolSize=4,
            symbolBrush="b",
        )
        
        self.plot_graph2 = pg.PlotWidget()
        layout.addWidget(self.plot_graph2)
        self.plot_graph2.setBackground("w")
        self.plot_graph2.setTitle("Volume vs Time", color="b", size="20pt")
        styles = {"color": "black", "font-size": "18px"}
        self.plot_graph2.setLabel("left", "Area of Droplet", **styles)
        self.plot_graph2.setLabel("bottom", "Time (min)", **styles)
        self.plot_graph2.addLegend()
        self.plot_graph2.showGrid(x=True, y=True)
        self.plot_graph.setYRange(0, np.max(volume))
        self.timestamp = [0]
        self.j=0
        pen = pg.mkPen(color=(0, 255, 255), width=2)
        self.line2 = self.plot_graph2.plot(
            self.timestamp,
            volume,
            name="Chemostat 1",
            pen=pen,
            symbol="+",
            symbolSize=4,
            symbolBrush="b",
        )
        
        
        
        exit_button = QPushButton('Exit')
        exit_button.setStyleSheet("background-color:#e2757a;")
        #exit_button.clicked.connect(relayAllOn)
        exit_button.clicked.connect(QApplication.instance().quit)
        exit_button.clicked.connect(QApplication.closeAllWindows)
        exit_button.clicked.connect(QApplication.exit)
        layout.addWidget(exit_button, 12, 4, 1, 4)
        
        
    
    
    def set_Cycles(self, s):
        self.Cycles = int(s)
        self.pbar.setRange(0, self.Cycles)
        print(self.Cycles)
        
    
    def set_Period(self, s):
        self.CyclePeriod = float(s)
        print(self.CyclePeriod)

    
    
    def Start_experiment(self):
        for i in range(self.Cycles + 1):
            #print(self.CyclePeriod)
            # slowing down the loop 
            ic = f"dropletstack/test_0_10_2_00{i+133}.tiff"
            vol,fluor = image_pipeline(ic)
            volume.append(vol)
            fluorescence.append(fluor)
            if self.timestamp.append(self.timestamp[-1] + self.CyclePeriod)
            self.pbar.setValue(i)
            self.plot_graph.setYRange(0, np.max(fluorescence))
            self.plot_graph2.setYRange(0, np.max(volume))
            self.line1.setData(self.timestamp[1:], fluorescence[1:])
            self.line2.setData(self.timestamp[1:], volume[1:])
            print(fluorescence, self.timestamp)
            sleep(self.CyclePeriod)
        
                
        
        
        
        
    def stack2UI(self):
        layout = QGridLayout()
        self.stack2.setLayout(layout)
        self.setStyleSheet("background-color: #f9f1e2;")
        self.setStyleSheet("QLabel{font-size: 15pt;}")
        My_Font = QFont("San Francisco", 12)
        
        rows = 8
        columns = 4
        button_count = 30

        for i in range(24):
            valve_switches[i] = AnimatedToggle( checked_color="#68C16E", pulse_checked_color="#44FFB000")
            valve_name = valve_names[i+1]
            #valve_states[valve_name] = True
            valve_switches[i].setCheckable(True)
            valve_switches[i].setChecked(True)
            #valve_switches[i].stateChanged.connect(self.the_button_was_toggled)
            #button.clicked.connect(partial(update_valve_state, valve_name))
            label = QLabel(valve_names[i+1])
            layout.addWidget(label, (i % rows), 2 * (i // rows), alignment = QtCore.Qt.AlignmentFlag.AlignRight)
            layout.addWidget(valve_switches[i], (i % rows), 2 * (i // rows) + 1, alignment = QtCore.Qt.AlignmentFlag.AlignLeft)
        
        
        #ALL A
        valve_switches[25] = SwitchControl(bg_color="#777777", circle_color="#fcfbea", active_color="#d22ed1", animation_curve=QtCore.QEasingCurve.InOutCubic, animation_duration=100,  checked=False, change_cursor=False)
        #valve_switches[25].setCheckable(True)
        valve_switches[25].setChecked(False)
        valve_switches[25].stateChanged.connect(self.TurnA)
        label = QLabel(valve_names[25])
        layout.addWidget(label, 9, 0, alignment=QtCore.Qt.AlignmentFlag.AlignRight)
        layout.addWidget(valve_switches[25], 9, 1, alignment=QtCore.Qt.AlignmentFlag.AlignLeft)
        
        #ALL B
        valve_switches[26] = SwitchControl(bg_color="#777777", circle_color="#fcfbea", active_color="#d22ed1", animation_curve=QtCore.QEasingCurve.InOutCubic, animation_duration=100,  checked=False, change_cursor=False)
        #valve_switches[26].setCheckable(True)
        valve_switches[26].setChecked(False)
        valve_switches[26].stateChanged.connect(self.TurnB)
        label = QLabel(valve_names[26])
        layout.addWidget(label, 9, 2, alignment=QtCore.Qt.AlignmentFlag.AlignRight)
        layout.addWidget(valve_switches[26], 9, 3, alignment=QtCore.Qt.AlignmentFlag.AlignLeft)
        
        #ALL C
        valve_switches[27] = SwitchControl(bg_color="#777777", circle_color="#fcfbea", active_color="#d22ed1", animation_curve=QtCore.QEasingCurve.InOutCubic, animation_duration=100,  checked=False, change_cursor=False)
        #valve_switches[27].setCheckable(True)
        valve_switches[27].setChecked(False)
        valve_switches[27].stateChanged.connect(self.TurnC)
        label = QLabel('All C')
        layout.addWidget(label, 9, 4, alignment=QtCore.Qt.AlignmentFlag.AlignRight)
        layout.addWidget(valve_switches[27], 9, 5, alignment=QtCore.Qt.AlignmentFlag.AlignLeft)
            
        


        ############Pulse generator
        button1 = QPushButton('123')
        button1.setText("Generate Droplet")
        #button1.clicked.connect(generatePulse)
        layout.addWidget(button1, 12, 0, alignment=QtCore.Qt.AlignmentFlag.AlignRight)



        exit_button = QPushButton('Exit')
        exit_button.setStyleSheet("background-color:#e2757a;")
        #exit_button.clicked.connect(relayAllOn)
        exit_button.clicked.connect(QApplication.instance().quit)
        exit_button.clicked.connect(QApplication.closeAllWindows)
        exit_button.clicked.connect(QApplication.exit)
        layout.addWidget(exit_button, 12, 2, 1, 4)
        
        
        voltagevalues = QComboBox()
        voltagevalues.addItems(['100', '200', '300', '400', '500', '600', '700'])
        voltagevalues.currentTextChanged.connect(set_voltage)
        label = QLabel('Operating voltage in V')
        layout.addWidget(label, 0, 8)
        layout.addWidget(voltagevalues, 1, 8, 1, 3)
        
        self.TurnOnVolts = QPushButton('Volts OFF')
        self.TurnOnVolts.setStyleSheet("background-color: #bb283a;")
        self.TurnOnVolts.setCheckable(True)
        self.TurnOnVolts.setChecked(False)
        self.TurnOnVolts.clicked.connect(self.alter_state)
        layout.addWidget(self.TurnOnVolts, 3, 8, 1, 3)
        
    
    
    

    def alter_state(self, checked):
        if checked == True:
            self.TurnOnVolts.setText('Volts ON')
            self.TurnOnVolts.setStyleSheet("background-color: #2ac555;")
            SendArduinoTrigger()
        else:
            self.TurnOnVolts.setText('Volts OFF')
            self.TurnOnVolts.setStyleSheet("background-color: #bb283a;")
            
            
            
            
            
    def TurnA(self, checked):
        if checked == 2:
            for i in range(8):
                valve_switches[i].setChecked(True)
        elif checked == 0:
            for i in range(8):
                valve_switches[i].setChecked(False)
    
    
    def TurnB(self, checked):
        if checked == 2:
            for i in range(8,16):
                valve_switches[i].setChecked(True)
        elif checked == 0:
            for i in range(8,16):
                valve_switches[i].setChecked(False)
    
    
    def TurnC(self, checked):
        print('toggled')
        if checked == 2:
            for i in range(16,24):
                valve_switches[i].setChecked(True)
                
        elif checked == 0:
            for i in range(16,24):
                valve_switches[i].setChecked(False)
                
                
                
                
                
    def the_button_was_toggled(self, checked):
        print("Checked?", checked)


    def activate_tab_1(self):
        self.stacklayout.setCurrentIndex(0)

    def activate_tab_2(self):
        self.stacklayout.setCurrentIndex(1)
        
        
        
		
    def display(self,i):
        self.Stack.setCurrentIndex(i)
    
    
    
		
def main():
    app = QApplication(sys.argv)
    ex = stackedApp()
    sys.exit(app.exec_())
	
if __name__ == '__main__':
   main()

3.0
7


C:\Users\amogh\AppData\Local\Temp\ipykernel_27976\3545142310.py:75: UserWarning: Output.tiff is a boolean image: setting True to 255 and False to 0. To silence this warning, please convert the image using img_as_ubyte.
  io.imsave('Output.tiff', binary_mask)


[0, 9541.320711049833] [0, 3.0]
[0, 9541.320711049833, 9521.9818370268] [0, 3.0, 6.0]
[0, 9541.320711049833, 9521.9818370268, 9502.803264094955] [0, 3.0, 6.0, 9.0]
[0, 9541.320711049833, 9521.9818370268, 9502.803264094955, 9490.346846623332] [0, 3.0, 6.0, 9.0, 12.0]
[0, 9541.320711049833, 9521.9818370268, 9502.803264094955, 9490.346846623332, 9457.446283867761] [0, 3.0, 6.0, 9.0, 12.0, 15.0]
[0, 9541.320711049833, 9521.9818370268, 9502.803264094955, 9490.346846623332, 9457.446283867761, 9444.076344246774] [0, 3.0, 6.0, 9.0, 12.0, 15.0, 18.0]
[0, 9541.320711049833, 9521.9818370268, 9502.803264094955, 9490.346846623332, 9457.446283867761, 9444.076344246774, 9421.074510587296] [0, 3.0, 6.0, 9.0, 12.0, 15.0, 18.0, 21.0]
[0, 9541.320711049833, 9521.9818370268, 9502.803264094955, 9490.346846623332, 9457.446283867761, 9444.076344246774, 9421.074510587296, 9403.87093058199] [0, 3.0, 6.0, 9.0, 12.0, 15.0, 18.0, 21.0, 24.0]


SystemExit: 0

C:\ProgramData\Anaconda3\envs\bootcamp\Lib\site-packages\IPython\core\interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
